### 1. Library Imports and Setup
This block imports all the necessary tools for data manipulation, preprocessing, and model training:
- **Core Libraries**: `numpy` and `pandas` for numerical operations and managing data structures (DataFrames). `math` for standard mathematical functions.
- **Preprocessing Tools**:
    - `MinMaxScaler`: To normalize numerical features into a specific range (usually 0 to 1).
    - `OrdinalEncoder` / `OneHotEncoder` / `LabelEncoder`: Different techniques to convert categorical text data into a numerical format that machine learning models can understand.
- **Model Utilities**: `train_test_split` to divide the dataset into training and testing sets to evaluate model performance effectively.


In [1]:
import numpy as np
import pandas as pd
import math
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split


### 2. Loading Credit Approval Dataset and Defining Feature Types
In this step, we read the raw data file and identify continuous numerical attributes:
- **`pd.read_csv()`**: Reads the `./crx.data` file with specific configurations:
  - `header=None`: Treats the first line as data because the CSV file does not contain column headers.
  - `na_values="?"`: Automatically converts all missing value indicators (`?`) in the raw file into standard `NaN` (Not a Number) values.
- **`cont_attr`**: Defines a Python list containing the zero-based column indices (`1, 2, 7, 10, 13, 14`) that represent continuous (numerical) features in the Credit Approval dataset.


In [2]:
raw_data = pd.read_csv("./crx.data", header=None, na_values="?")

cont_attr = [1, 2, 7, 10, 13, 14]

### 3. Handling Missing Values (Data Imputation)
In this step, we loop through all 16 attributes and fill missing values (`NaN`) based on feature type:
- **`for col in range(16):`**: Iterates across each column index from `0` to `15`.
- **Continuous Features (`if col in cont_attr:`):**
  - Calculates the column mean (`.mean()`).
  - Fills all missing entries with this average value using `.fillna()`.
- **Categorical Features (`else:`):**
  - Finds the most frequent category using the column mode (`.mode()[0]`).
  - Fills missing entries with this frequent value.
- **Verification (`print(raw_data.isna().sum())`):** Displays the count of missing values per column to confirm that no `NaN` entries remain in the dataset.


In [3]:
for col in range(16):
    if col in cont_attr:
        _mean = raw_data[col].mean()
        raw_data[col] = raw_data[col].fillna(_mean)
    else:
        _mode = raw_data[col].mode()[0]
        raw_data[col] = raw_data[col].fillna(_mode)

print(raw_data.isna().sum())

0     0
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     0
10    0
11    0
12    0
13    0
14    0
15    0
dtype: int64


### 4. Continuous Feature Normalization (MinMaxScaler)
In this step, we scale all continuous numerical columns into a standard range between 0 and 1:
- **`MinMaxScaler()`**: Initializes a min-max scaling object to normalize feature values.
- **`fit_transform()`**: Computes the min and max values of the specified continuous columns (`cont_attr`) and transforms the data accordingly into a NumPy array (`x_scaled`).
- **DataFrame Reconstruction**: Converts `x_scaled` back into a Pandas DataFrame (`normalized`) matching the original indices and column names.
- **Updating Dataset**: Replaces the original continuous columns in `raw_data` with the newly scaled values and displays the updated DataFrame.


In [4]:
min_max_scaler = MinMaxScaler()
x = raw_data[cont_attr].values
x_scaled = min_max_scaler.fit_transform(x)

normalized = pd.DataFrame(x_scaled, columns=cont_attr, index=raw_data.index)
raw_data[cont_attr] = normalized
raw_data

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,b,0.256842,0.000000,u,g,w,v,0.043860,t,t,0.014925,f,g,0.1010,0.00000,+
1,a,0.675489,0.159286,u,g,q,h,0.106667,t,t,0.089552,f,g,0.0215,0.00560,+
2,a,0.161654,0.017857,u,g,q,h,0.052632,t,f,0.000000,f,g,0.1400,0.00824,+
3,b,0.211729,0.055000,u,g,w,v,0.131579,t,t,0.074627,t,g,0.0500,0.00003,+
4,b,0.096541,0.200893,u,g,w,v,0.060000,t,f,0.000000,f,s,0.0600,0.00000,+
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
685,b,0.110226,0.360179,y,p,e,h,0.043860,f,f,0.000000,f,g,0.1300,0.00000,-
686,a,0.134135,0.026786,u,g,c,v,0.070175,f,t,0.029851,t,g,0.1000,0.00394,-
687,a,0.172932,0.482143,y,p,ff,ff,0.070175,f,t,0.014925,t,g,0.1000,0.00001,-
688,b,0.062707,0.007321,u,g,aa,v,0.001404,f,f,0.000000,f,g,0.1400,0.00750,-


### 5. Target Label Encoding (OrdinalEncoder)
In this step, we convert the categorical target column (column index `15`) into integer values:
- **`OrdinalEncoder(dtype=np.int32)`**: Initializes an ordinal encoder that assigns distinct integer codes (as 32-bit integers) to each unique category (e.g., `+` and `-` become `0` and `1`).
- **`fit_transform(raw_data[[15]])`**: Learns the unique classes from column `15` (passed as a 2D DataFrame slice using double brackets `[[15]]`) and transforms them into numerical labels.
- **Updating Target Column**: Replaces the original string labels in column `15` with the encoded integer values.


In [5]:
ord_enc = OrdinalEncoder(dtype = np.int32)
raw_data[15] = ord_enc.fit_transform(raw_data[[15]])


### 6. Encoding Categorical Features (One-Hot Encoding) and Building Feature Matrix
In this step, we convert all remaining discrete/categorical attributes into binary columns and assemble the final feature matrix:
- **`dis_attr`**: Defines the zero-based column indices (`0, 3, 4, 5, 6, 8, 9, 11, 12`) that represent discrete (categorical) attributes.
- **`pd.get_dummies(..., dtype=int)`**: Generates one-hot encoded binary columns (`0` or `1`) for each category in the discrete attributes.
  - Setting `dtype=int` ensures integers are used instead of the default boolean `True`/`False` values.
- **`.drop(dis_attr, axis=1)`**: Removes the original raw discrete columns from `raw_data` since they are replaced by their encoded versions.
- **`.join(one_hat)`**: Combines the remaining continuous/encoded columns with the one-hot binary columns to create the final feature matrix `X`.


In [6]:
dis_attr = [0, 3, 4, 5, 6, 8, 9, 11, 12]
one_hat = pd.get_dummies(raw_data[dis_attr], dtype=int)
raw_data = raw_data.drop(dis_attr, axis = 1)
X = raw_data.join(one_hat)

### 7. Target Separation and Train-Test Split
In this step, we separate the target variable and divide the dataset for training and evaluation:
- **`X.pop(15)`**: Removes the encoded target column (index `15`) from the feature matrix `X` and converts it into a NumPy array `Y`.
- **`np.array(X)`**: Converts the remaining feature DataFrame into a NumPy array for compatibility with Scikit-Learn models.
- **Shape Verification**: Prints for compatibility with Scikit-Learn models.
- **Shape Verification**: Prints the dimensions of_features)` and `(690,)`).
- **`train_test_split(..., test_size=0.2)`**: Randomly splits the data into 80% training and 20% testing subsets:
  - `x_train` / `Y_train`: Used to train the model.
  - `X_test` / `Y_test`: Used to evaluate model performance on unseen data.
- Prints the shapes of the test sets to confirm the 20% split.


In [7]:
Y = np.array(X.pop(15))
X = np.array(X)
print(X.shape, Y.shape)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2)
print(X_test.shape, Y_test.shape)

(690, 46) (690,)
(138, 46) (138,)
